<a href="https://sigmoidal.ai">
  <img src="https://raw.githubusercontent.com/carlosfab/blog-sigmoidal/main/_assets/logo_sigmoidal.png" alt="Sigmoidal" width="300">
</a>

# Grad-CAM: Visualizando o que uma CNN Enxerga

**Autor:** Carlos Melo — [sigmoidal.ai](https://sigmoidal.ai)

## Setup: Ambiente e Treinamento do Modelo

Nesta seção, configuramos o ambiente e treinamos rapidamente um modelo com fine-tuning no Oxford Flowers 102. Se você já fez o notebook de Transfer Learning, pode pular para a seção de Grad-CAM.

### Imports e configuração do dispositivo

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

### Transforms e dataset Oxford Flowers 102

In [ ]:
# Transforms
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Download dos datasets
train_dataset = torchvision.datasets.Flowers102(
    root="./data", split="train", download=True, transform=train_transform
)
val_dataset = torchvision.datasets.Flowers102(
    root="./data", split="val", download=True, transform=test_transform
)
test_dataset = torchvision.datasets.Flowers102(
    root="./data", split="test", download=True, transform=test_transform
)

print(f"Treino: {len(train_dataset)} imagens")
print(f"Valida\u00e7\u00e3o: {len(val_dataset)} imagens")
print(f"Teste: {len(test_dataset)} imagens")

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

### Função auxiliar e modelo com fine-tuning

In [ ]:
def desnormalizar(img):
    """Reverte a normaliza\u00e7\u00e3o ImageNet para exibir a imagem."""
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img = img * std + mean
    img = torch.clamp(img, 0, 1)
    return img

In [ ]:
# Carregar ResNet18 pre-treinada
modelo = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

# Congelar tudo
for param in modelo.parameters():
    param.requires_grad = False

# Descongelar layer4
for param in modelo.layer4.parameters():
    param.requires_grad = True

# Trocar a camada fc para 102 classes
modelo.fc = nn.Linear(modelo.fc.in_features, 102)

total_params = sum(p.numel() for p in modelo.parameters())
treinaveis = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
print(f"Par\u00e2metros totais: {total_params:,}")
print(f"Par\u00e2metros trein\u00e1veis: {treinaveis:,} ({100*treinaveis/total_params:.1f}%)")

modelo = modelo.to(device)

### Treinamento com fine-tuning (15 epochs)

In [ ]:
def treinar(model, train_loader, val_loader, criterion, optimizer, num_epochs):
    train_losses = []
    val_accuracies = []

    for epoch in range(num_epochs):
        # Treino
        model.train()
        running_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        avg_loss = running_loss / len(train_loader)
        train_losses.append(avg_loss)

        # Valida\u00e7\u00e3o
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_acc = 100 * correct / total
        val_accuracies.append(val_acc)

        print(f"Epoca {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f} - Val Acc: {val_acc:.1f}%")

    return train_losses, val_accuracies

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam([
    {"params": modelo.layer4.parameters(), "lr": 1e-4},
    {"params": modelo.fc.parameters(), "lr": 1e-3},
])

print("Treinando com fine-tuning (layer4 + fc)")
print()
ft_losses, ft_accs = treinar(modelo, train_loader, val_loader, criterion, optimizer, num_epochs=15)

print(f"\nMelhor acur\u00e1cia na valida\u00e7\u00e3o: {max(ft_accs):.1f}%")

---

## Grad-CAM: Visualizando a Atenção da Rede

Grad-CAM (Gradient-weighted Class Activation Mapping) permite visualizar quais regiões da imagem a rede utiliza para tomar sua decisão. A técnica captura os gradientes que fluem para a última camada convolucional e os usa para ponderar os mapas de ativação.

### Implementação da classe GradCAM

Registramos hooks na última camada convolucional para capturar as ativações (forward) e os gradientes (backward). Com isso, calculamos um mapa de calor que destaca as regiões mais relevantes para a classe predita.

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None

        # Registrar hooks
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_img, class_idx=None):
        self.model.eval()
        output = self.model(input_img)

        if class_idx is None:
            class_idx = output.argmax(dim=1).item()

        self.model.zero_grad()
        output[0, class_idx].backward()

        # Pesos: media global dos gradientes por canal
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)

        # Mapa de ativa\u00e7\u00e3o ponderado
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = torch.relu(cam)

        # Normalizar entre 0 e 1
        cam = cam - cam.min()
        if cam.max() > 0:
            cam = cam / cam.max()

        return cam.squeeze().cpu().numpy(), class_idx

### Função de visualização

A função abaixo sobrepõe o mapa de calor do Grad-CAM na imagem original, mostrando três visões: a imagem pura, o mapa de ativação e a sobreposição dos dois.

In [ ]:
def mostrar_gradcam(model, img_tensor, target_layer, label_real=None):
    gradcam = GradCAM(model, target_layer)

    img_gpu = img_tensor.unsqueeze(0).to(device)
    cam, pred_class = gradcam.generate(img_gpu)

    # Redimensionar o CAM para o tamanho da imagem
    cam_resized = F.interpolate(
        torch.tensor(cam).unsqueeze(0).unsqueeze(0),
        size=(224, 224),
        mode="bilinear",
        align_corners=False
    ).squeeze().numpy()

    # Imagem original desnormalizada
    img_show = desnormalizar(img_tensor).permute(1, 2, 0).numpy()

    plt.figure(figsize=(10, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(img_show)
    plt.title("Imagem original")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(cam_resized, cmap="jet")
    plt.title("Grad-CAM")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(img_show)
    plt.imshow(cam_resized, cmap="jet", alpha=0.5)
    title = f"Pred: {pred_class}"
    if label_real is not None:
        title += f" | Real: {label_real}"
    plt.title(title)
    plt.axis("off")

    plt.tight_layout()
    plt.show()

### Grad-CAM em exemplos do conjunto de teste

Vamos aplicar o Grad-CAM em imagens do teste para ver onde a rede foca sua atenção ao classificar cada flor.

In [ ]:
# Pegar imagens do teste
test_images, test_labels = next(iter(test_loader))

# A ultima camada convolucional da ResNet18 esta no final da layer4
target_layer = modelo.layer4[-1].conv2

for i in range(6):
    mostrar_gradcam(modelo, test_images[i], target_layer, label_real=test_labels[i].item())

### Comparando acertos e erros

Onde a rede foca quando acerta? E quando erra? Essa comparação revela se os erros estão ligados a regiões irrelevantes da imagem (fundo, folhas) em vez da flor em si.

In [ ]:
# Coletar predi\u00e7\u00f5es para encontrar acertos e erros
modelo.eval()
acertos = []
erros = []

with torch.no_grad():
    for images, labels in test_loader:
        images_gpu = images.to(device)
        outputs = modelo(images_gpu)
        _, preds = torch.max(outputs, 1)
        preds = preds.cpu()

        for i in range(len(labels)):
            if preds[i] == labels[i] and len(acertos) < 3:
                acertos.append((images[i], labels[i].item()))
            elif preds[i] != labels[i] and len(erros) < 3:
                erros.append((images[i], labels[i].item()))

            if len(acertos) >= 3 and len(erros) >= 3:
                break

        if len(acertos) >= 3 and len(erros) >= 3:
            break

print(f"Acertos coletados: {len(acertos)}")
print(f"Erros coletados: {len(erros)}")

In [ ]:
print("ACERTOS:")
target_layer = modelo.layer4[-1].conv2
for img, label in acertos:
    mostrar_gradcam(modelo, img, target_layer, label_real=label)

In [ ]:
print("ERROS:")
for img, label in erros:
    mostrar_gradcam(modelo, img, target_layer, label_real=label)